In [3]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['JuegosBigData'] 
coleccion = db['JugandoConPaginas'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


In [7]:
import os
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")

NOMBRE_GRUPO = "Agrotech"
datos_finales = []
driver = None

options = Options()
options.binary_location = "/usr/bin/google-chrome"
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")

try:
    driver = webdriver.Chrome(options=options)
    print("Navegador iniciado correctamente")

    url = "https://climatologia.meteochile.gob.cl/application/anual/indiceUvbMaximoAnual/290004/2025"
    driver.get(url)

    time.sleep(15)

    filas = driver.find_elements(By.TAG_NAME, "tr")

    for fila in filas:
        columnas = fila.find_elements(By.TAG_NAME, "td")
        if len(columnas) >= 2:
            datos_finales.append({
                "dato_a": columnas[0].text,
                "dato_b": columnas[1].text,
                "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                "grupo": Agrotech,
                "fuente": "MeteoChile"
            })

    print(f"Extracción terminada: {len(datos_finales)} registros.")

except Exception as e:
    print("Error en Selenium:", e)

finally:
    if driver is not None:
        try:
            driver.quit()
        except:
            pass

try:
    client = MongoClient("database", 27017, serverSelectionTimeoutMS=5000)
    db = client["ClimaBigData"]
    coleccion = db["MeteoChileUV"]

    if datos_finales:
        coleccion.insert_many(datos_finales)
        print("Datos cargados en MongoDB correctamente.")
    else:
        print("No hay datos para guardar.")

except Exception as e:
    print("Error en MongoDB:", e)

Navegador iniciado correctamente
Extracción terminada: 40 registros.
Datos cargados en MongoDB correctamente.
